# G07 — Selected two-class from-scratch LDM with a frozen SD 2.1 VAE

Generator **G07 (`07_ldm_sdvae_extra1361`)** trains a class-conditional Keras U-Net from random initialization in the latent space of the frozen Stable Diffusion 2.1 `AutoencoderKL`. Only the VAE supplies the representation and decoder; no Stable Diffusion U-Net weights are reused. This distinction makes G07 a from-scratch denoiser with a pretrained latent codec, rather than an SD 2.1 fine-tune.

The notebook produces complete positive and negative pools: 4,083 raw samples per class are filtered to 1,361 images per class under `data/synthetic/07_ldm_sdvae_extra1361/{positive,negative}/`. The unified generator benchmark and the unified generator benchmark subsequently selected G07 as the from-scratch generator for the downstream study. That selection is external to this notebook and should not be inferred from a single metric shown here.

Reproducibility is supported by two manifests: `latents/latents_manifest.json` binds the encoded training/validation data to their metadata and VAE source, and `training_manifest.json` records the U-Net parameterization and terminal checkpoint. Models, caches, and logs live in `experiments/diffusers/07_ldm_sdvae_extra1361/`, while metrics, plots, and sustainability events live in `results/2_diffusers/07_ldm_sdvae_extra1361/`.


## 1. Runtime bootstrap and device policy

The bootstrap locates the repository, makes utility modules importable, configures XLA `libdevice` when available, reports the physical GPU inventory, and prints a dry-run resolution of the generation workers. Training and generation use separate child-process environments; generation can distribute indexed work across all eligible devices.

Device selection must remain external or runtime-resolved for portability. The hardware inventory is runtime metadata, not part of G07's scientific definition, and changing GPU hardware does not authorize changing checkpoints, seeds, or manifests.


In [1]:
# === Unified notebooks/ bootstrap ===
# Works from the project root and every subdirectory under notebooks/.
import sys as _sys
from pathlib import Path as _Path

def _find_mammo_root():
    # A repository marker must win over a directory that merely carries the
    # project name: this release lives inside a parent directory called
    # "MammoDiffusion" that also holds a separate successor project, so a
    # checkout without data/ must still resolve to the checkout itself.
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if (_candidate / "notebooks").is_dir() and (
            (_candidate / ".git").exists()
            or (_candidate / "configs").is_dir()
            or (_candidate / "data").is_dir()
        ):
            return _candidate
        # A directory is only the project because of its name when it also looks
        # like the project: a bare name can now be a symlink to another repository.
        if _candidate.name == "MammoDiffusion" and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("MammoDiffusion root not found from " + str(_Path.cwd()))

PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

# === End unified bootstrap ===

# Execution contract: offline and non-mutating unless deliberately opted into.
# See notebooks/utility/review_mode.py for the flags and the environment overrides.
import review_mode
ALLOW_NETWORK_ACCESS = False
INSTALL_DEPENDENCIES = False
ALLOW_PROCESSED_DOWNLOAD = False
review_mode.activate(
    allow_network=ALLOW_NETWORK_ACCESS,
    allow_dependency_install=INSTALL_DEPENDENCIES,
    allow_processed_download=ALLOW_PROCESSED_DOWNLOAD,
)

import os
import subprocess
import sys
from pathlib import Path

CUDA_ROOT = Path(os.environ.get("MAMMODIFFUSION_CUDA_ROOT", os.environ.get("CONDA_PREFIX", sys.prefix)))

libdevice_path = CUDA_ROOT / "nvvm" / "libdevice" / "libdevice.10.bc"
if libdevice_path.exists():
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={CUDA_ROOT}"

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print("Available physical GPUs:")
        print(result.stdout.strip())
except FileNotFoundError:
    print("nvidia-smi is unavailable.")

print("XLA_FLAGS:", os.environ.get("XLA_FLAGS", ""))

# Inherit CUDA visibility by default; optionally override it for training subprocesses.
TRAIN_GPU_VISIBLE_DEVICES = os.environ.get("MAMMODIFFUSION_TRAIN_GPU")

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Multi-GPU generation does not affect training.
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command

print("Inherited CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Requested GENERATION_GPU_DEVICES:", GENERATION_GPU_DEVICES)
from parallel_generation_utils import print_gpu_resolution_dry_run
print_gpu_resolution_dry_run(GENERATION_GPU_DEVICES, GENERATION_MAX_WORKERS)

Available physical GPUs:
0, NVIDIA GeForce RTX 3060, 12288 MiB
1, NVIDIA GeForce RTX 5060 Ti, 16311 MiB
XLA_FLAGS: 
Inherited CUDA_VISIBLE_DEVICES: None
Requested GENERATION_GPU_DEVICES: auto
Physical GPUs (nvidia-smi): ['0', '1']
Inherited CUDA_VISIBLE_DEVICES: None
Requested GPUs (--generation-gpus): auto
Resolved GPUs: ['0', '1']
Worker count: 2


['0', '1']

## 2. Dependency bootstrap

Dependency installation is disabled by default through `INSTALL_DEPENDENCIES=False`. When intentionally enabled while preparing a new environment, the cell installs the numerical, TensorFlow, PyTorch, Diffusers, and metric packages required by G07. It creates no scientific result. A publication rerun should retain the default in an already provisioned environment and record resolved versions whenever the dependency set is changed.


In [2]:
# INSTALL_DEPENDENCIES is set once in the bootstrap cell, together with the
# rest of the execution contract; the project environment should normally be
# provisioned from requirements.txt instead.
if INSTALL_DEPENDENCIES:
    %pip install -q pandas numpy matplotlib scikit-learn pillow gdown tensorflow scikit-image scipy psutil codecarbon torch torchvision torchmetrics torch-fidelity prdc diffusers transformers accelerate safetensors
else:
    print('Dependencies already provisioned; installation skipped.')

Dependencies already provisioned; installation skipped.


## 3. Project paths, subprocess logging, and phase flags

The setup cell defines canonical paths for the shared SD 2.1 snapshot, G07 models/checkpoints/latents/logs, class-specific synthetic outputs, result tables, figures, and EcoTracker records. It verifies every helper program before work begins. `run_and_stream` starts each helper from the repository root, persists combined stdout/stderr to a phase log, streams progress to the notebook, and raises on a non-zero exit code.

Every phase is a boolean that starts `False`: from-scratch training and generation run only when their flag is set, and latent preparation follows `RUN_TRAINING_PHASE`. The earlier bootstrap cell may still create canonical directories or validate shared assets. This design preserves resumable state and prevents an ordinary Run All from silently replacing a validated 150,000-step model or a complete two-class pool.


In [3]:
# Setup must precede the phase flags; this cell only defines paths/helpers.
if True:
    from pathlib import Path
    from tempfile import TemporaryDirectory
    import os
    import shutil
    import subprocess
    import sys
    import time
    import zipfile

    import gdown

    PROJECT_NAME = "MammoDiffusion"
    EXPERIMENT_NAME = "diffusers/07_ldm_sdvae_extra1361"
    RESULTS_STAGE_NAME = "2_diffusers/07_ldm_sdvae_extra1361"

    SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
    FORCE_MODEL_REDOWNLOAD = False
    PROJECT_ROOT_OVERRIDE = None

    def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
        if override is not None:
            root = Path(override).expanduser().resolve()
            if not root.exists():
                raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {root}")
            return root

        cwd = Path.cwd().resolve()
        for candidate in [cwd, *cwd.parents]:
            if candidate.name == project_name and (candidate / "notebooks").is_dir():
                return candidate
            has_notebooks = (candidate / "notebooks").exists() or (candidate / "notebooks").exists()
            if ((candidate / "data").exists() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
                return candidate
        for candidate in [
            cwd / project_name,
            Path("/content") / project_name,
            Path("/content/drive/MyDrive") / project_name,
            Path.home() / project_name,
        ]:
            if candidate.is_dir() and (candidate / "notebooks").is_dir():
                return candidate.resolve()
        raise FileNotFoundError("MammoDiffusion root not found.")

    PROJECT_ROOT = find_project_root()
    NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
    UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
    DATA_DIR = PROJECT_ROOT / "data"
    DATA_PROCESSED_DIR = DATA_DIR / "processed"
    EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
    SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / "pretrained_model"
    PRETRAINED_MODEL_DIR = SHARED_PRETRAINED_ROOT / "stable-diffusion-2-1-base"
    PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / "archives" / "stable-diffusion-2-1-base.zip"
    SD_VAE_MODEL_NAME_OR_PATH = PRETRAINED_MODEL_DIR
    MODELS_DIR = EXPERIMENT_DIR / "models"
    CHECKPOINTS_DIR = EXPERIMENT_DIR / "checkpoints_ldm"
    LATENTS_DIR = EXPERIMENT_DIR / "latents"
    LOGS_DIR = EXPERIMENT_DIR / "logs"
    RESULTS_DIR = PROJECT_ROOT / "results" / RESULTS_STAGE_NAME
    RESULTS_PLOTS_DIR = RESULTS_DIR / "plots"
    RESULTS_METRICS_DIR = RESULTS_DIR / "metrics"
    RESULTS_ECOTRACKER_DIR = RESULTS_DIR / "ecotracker"
    SYNTHETIC_NEW_DIR = DATA_DIR / "synthetic" / "07_ldm_sdvae_extra1361"
    SYNTHETIC_NEW_POS_DIR = SYNTHETIC_NEW_DIR / "positive"
    SYNTHETIC_NEW_NEG_DIR = SYNTHETIC_NEW_DIR / "negative"

    for directory in [
        EXPERIMENT_DIR, PRETRAINED_MODEL_ZIP_PATH.parent, MODELS_DIR, CHECKPOINTS_DIR, LATENTS_DIR, LOGS_DIR,
        RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR, RESULTS_ECOTRACKER_DIR,
        SYNTHETIC_NEW_POS_DIR, SYNTHETIC_NEW_NEG_DIR,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    HELPERS = [
        UTILITY_DIR / "prepare_sdvae_latents.py",
        UTILITY_DIR / "train_ldm.py",
        UTILITY_DIR / "evaluate_ldm.py",
        UTILITY_DIR / "generate_ldm.py",
                UTILITY_DIR / "sd_vae_utils.py",
    ]
    for helper in HELPERS:
        if not helper.exists():
            raise FileNotFoundError(helper)

    def run_and_stream(cmd, log_path, env=None):
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("Command:")
        print(" ".join(map(str, cmd)))
        print("Log:", log_path)
        with open(log_path, "w", encoding="utf-8") as log_file:
            proc = subprocess.Popen(
                [str(x) for x in cmd],
                cwd=str(PROJECT_ROOT),
                stdout=log_file,
                stderr=subprocess.STDOUT,
                env=env or os.environ.copy(),
                start_new_session=True,
            )
        print("PID:", proc.pid)
        with open(log_path, "r", encoding="utf-8", errors="replace") as log_file:
            while proc.poll() is None:
                line = log_file.readline()
                if line:
                    print(line, end="", flush=True)
                else:
                    time.sleep(0.5)
            for line in log_file:
                print(line, end="", flush=True)
        print("Return code:", proc.returncode)
        if proc.returncode != 0:
            raise RuntimeError(f"Command failed; check the log: {log_path}")

    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
    print("RESULTS_DIR:", RESULTS_DIR)
    print("PRETRAINED_MODEL_DIR:", PRETRAINED_MODEL_DIR)
    print("Filtered output:", SYNTHETIC_NEW_DIR)

PROJECT_ROOT: /mnt/MammoDiffusion/MammoDiffusion
EXPERIMENT_DIR: /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/07_ldm_sdvae_extra1361
RESULTS_DIR: /mnt/MammoDiffusion/MammoDiffusion/results/2_diffusers/07_ldm_sdvae_extra1361
PRETRAINED_MODEL_DIR: /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base
Filtered output: /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/07_ldm_sdvae_extra1361


In [4]:
# EXPLICIT_PHASE_FLAGS_V1
# One boolean per phase. Every flag is False, so an ordinary Run All reads the
# artifacts already on disk and reports them: it never retrains and never
# regenerates. To do real work, set the flags for the phases you intend to run
# and execute the notebook top to bottom. Flags are independent -- filtering can
# be redone without regenerating, latents rebuilt without retraining.
#
#   RUN_LATENT_ENCODING_PHASE  encode the VAE latents that training consumes; heavy only when they are missing or stale
#   RUN_TRAINING_PHASE         train the model (hours of GPU)
#   RUN_GENERATION_PHASE       sample a full RAW image pool from the selected checkpoint
#   RUN_EVALUATION_PHASE       score checkpoints and record the selection
#   RUN_FILTER_PHASE           re-run the adaptive filter over the existing RAW pool
#   RUN_VALIDATION_PHASE       RAW-vs-filtered comparison; keeps its own content-aware cache
#
# Leaving a flag False asserts that the phase's artifact is already complete.
# The cells below check that claim and raise if it does not hold, rather than
# reporting a number they did not verify.
RUN_LATENT_ENCODING_PHASE = False
RUN_TRAINING_PHASE = False
RUN_GENERATION_PHASE = False
RUN_EVALUATION_PHASE = False  # the saved selection is consumed, not recomputed
RUN_FILTER_PHASE = False
RUN_VALIDATION_PHASE = False

## 4. Split-metadata preflight

The cell requires the canonical training, validation, and test CSV files and prints sample counts by label for each split. Missing metadata stops execution before any model or latent cache is opened. Training and latent preparation use only training and validation records; the test split is checked only for structural completeness and remains reserved for final classifier evaluation.

This is a lightweight structural check. The latent-preparation helper performs the stronger image-path and metadata-signature validation, while patient-level split independence remains an upstream preprocessing invariant.


In [5]:
import pandas as pd

required = [
    DATA_PROCESSED_DIR / "metadata" / "train.csv",
    DATA_PROCESSED_DIR / "metadata" / "val.csv",
    ]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f"Missing preprocessed dataset: {path}")

for split in ["train", "val"]:
    df = pd.read_csv(DATA_PROCESSED_DIR / "metadata" / f"{split}.csv")
    print(split, len(df), df["label"].value_counts().to_dict())

train 2041 {0: 1701, 1: 340}
val 437 {0: 364, 1: 73}


## 5. Local Stable Diffusion 2.1 snapshot

The code reuses the local SD 2.1 directory when all required Diffusers components and standardized weight filenames are present. Otherwise it downloads the configured ZIP, validates the archive and component structure in a temporary directory, copies the model into the shared pretrained location, and verifies the result. Missing or malformed weights stop execution.

G07 subsequently loads only the VAE for latent encoding and decoding; the text encoder and SD U-Net are not optimization initializers for G07. Archive/structure checks prevent partial loading, but a publication hand-off should also retain the repository-level content identity of the shared snapshot rather than relying on the download identifier alone.


In [6]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}

def download_zip(drive_id, destination, force=False):
    review_mode.require_processed_download(
        'downloading a shared asset archive from Google Drive')
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Model archive already present:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Invalid model download: {destination}")

def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} not found under {root}")

def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)

def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)

def create_model_weight_copies(model_dir):
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Source weight not found: {source_path}")
        shutil.copy2(source_path, target_path)

def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Stable Diffusion 2.1 model already ready.")
        return PRETRAINED_MODEL_DIR.absolute()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "Diffusers model")
        if PRETRAINED_MODEL_DIR.is_symlink() or PRETRAINED_MODEL_DIR.is_file():
            PRETRAINED_MODEL_DIR.unlink()
        elif PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Stable Diffusion 2.1 model is incomplete after extraction.")
    return PRETRAINED_MODEL_DIR.absolute()

LOCAL_MODEL_DIR = prepare_pretrained_model()
SD_VAE_MODEL_NAME_OR_PATH = LOCAL_MODEL_DIR
print("Local model:", LOCAL_MODEL_DIR)

Stable Diffusion 2.1 model already ready.
Local model: /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base


## 6. SD-VAE load preflight

This cell resolves the local `AutoencoderKL`, loads it once, prints its device, numerical precision, and scaling factor, then releases it and clears the CUDA cache. The operation fails early if the VAE cannot be decoded by the installed Diffusers stack.

The scaling factor is part of the latent representation and must remain consistent across encoding, training, evaluation, and generation. This preflight produces no learned G07 parameters; it validates the frozen codec used by all later phases.


In [7]:
import torch

sys.path.insert(0, str(NOTEBOOKS_DIR))
from sd_vae_utils import load_sd_vae, resolve_sd_vae_model

SD_VAE_MODEL = resolve_sd_vae_model(PROJECT_ROOT, SD_VAE_MODEL_NAME_OR_PATH)
print("SD_VAE_MODEL:", SD_VAE_MODEL)

# Lightweight check: load only the VAE, print its scaling factor, then release memory.
vae, vae_device, vae_dtype, scaling_factor = load_sd_vae(SD_VAE_MODEL)
print("VAE device:", vae_device)
print("VAE dtype:", vae_dtype)
print("VAE scaling_factor:", scaling_factor)
del vae
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SD_VAE_MODEL: /mnt/MammoDiffusion/MammoDiffusion/notebooks/pretrained_model/stable-diffusion-2-1-base


bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/site-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/site-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/ctypes/__init__.py", line 454, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


VAE device: cuda
VAE dtype: torch.float16
VAE scaling_factor: 0.18215


## 7. Manifest-bound SD-VAE latent preparation

`prepare_sdvae_latents.py` reconstructs canonical image paths, applies three deterministic mild augmentations to each positive training image, and encodes the augmented training set and unaugmented validation set in batches of four. It saves `latents_train.npz`, `latents_val.npz`, per-channel normalization statistics in `latent_stats.npz`, and the VAE metadata under the G07 experiment. Test images are never encoded for training.

`latents_manifest.json` (schema version 2) records the VAE path, SHA-256 hashes of the train CSV, validation CSV, and augmentation metadata, sample counts, image/latent dimensions, and augmentation count. Cached latents are reused only when all files exist and the stored manifest exactly matches the newly computed identity; `FORCE_LATENTS_RECOMPUTE=True` explicitly invalidates that cache. The log `prepare_sdvae_latents.log` preserves the full preprocessing trace.


In [8]:
SDVAE_BATCH_SIZE = 4
FORCE_LATENTS_RECOMPUTE = False

prepare_cmd = [
    sys.executable,
    str(UTILITY_DIR / "prepare_sdvae_latents.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--batch-size", str(SDVAE_BATCH_SIZE),
]
prepare_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
if FORCE_LATENTS_RECOMPUTE:
    prepare_cmd.append("--force-recompute")

if RUN_LATENT_ENCODING_PHASE:
    run_and_stream(prepare_cmd, LOGS_DIR / "prepare_sdvae_latents.log")
else:
    print("Reusing the existing latent cache: RUN_LATENT_ENCODING_PHASE = False.")

Reusing the existing latent cache: RUN_LATENT_ENCODING_PHASE = False.


## 8. From-scratch conditional U-Net training

When `RUN_TRAINING_PHASE` is set, `train_ldm.py` consumes the manifest-validated latents, initializes the baseline conditional U-Net from scratch, and optimizes for 150,000 steps with checkpoints every 5,000 steps. `--skip-latent-encoding` prevents a second representation build, and `--resume-from-latest` continues from the highest compatible Keras checkpoint. The stream is saved to `logs/ldm_train_sdvae.log`.

Resume restores the saved model, inferred global step, and loss history; it should not be assumed bitwise equivalent to an uninterrupted run because optimizer continuity is not established by the notebook. Terminal models and periodic checkpoints are retained, and `training_manifest.json` records the default v2/epsilon parameterization and final model source. Validation checkpoint selection remains downstream of training loss.


In [9]:
# IDEMPOTENT_GUARD_V1:training
if RUN_TRAINING_PHASE:
    TOTAL_STEPS = 150_000
    CHECKPOINT_EVERY = 5_000
    LOG_EVERY = 20
    RESUME_FROM_LATEST = True

    train_cmd = [
        sys.executable,
        str(UTILITY_DIR / "train_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--total-steps", str(TOTAL_STEPS),
        "--checkpoint-every", str(CHECKPOINT_EVERY),
        "--log-every", str(LOG_EVERY),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--skip-latent-encoding",
    ]
    if RESUME_FROM_LATEST:
        train_cmd.append("--resume-from-latest")

    run_and_stream(train_cmd, LOGS_DIR / "ldm_train_sdvae.log", env=training_subprocess_env())

## 9. Validation-only checkpoint evaluation

The `evaluate_ldm.py` sweep generates 100 validation samples per class for every eligible checkpoint using 100 denoising steps, guidance scale 1.5, and the same frozen SD VAE. It computes FID, Inception Score, and PRDC and selects the checkpoint by positive-class FID with the registered tie-breaker.

The saved validation checkpoint selection is reused by default. Set `RUN_EVALUATION_PHASE=True` with `EVAL_FORCE_RECOMPUTE=True` to run the validation sweep again, which forces new manifests; sweep directories written before per-directory generation manifests existed are not rerun automatically.

The notebook validates `evaluation/best_checkpoint.json`, reroots its periodic checkpoint filename through the current project directory, and consumes that file rather than the mutable `ldm_unet_best_eval.keras` convenience copy. The small generated sample count and generic Inception representation make close differences uncertain; reusing the stored decision verifies checkpoint consumption, not metric recomputation.

In [10]:
# IDEMPOTENT_GUARD_V1:evaluation
import json

EVAL_MIN_STEP = 1_000
N_GEN_PER_CLASS = 100
EVAL_SAMPLE_STEPS = 100
EVAL_GUIDANCE_SCALE = 1.5
EVAL_INCEPTION_BATCH = 8
EVAL_DECODE_ON_CPU = False
EVAL_ECO_TRACK = True
EVAL_FORCE_RECOMPUTE = False  # ignore cached metrics and reevaluate

if RUN_EVALUATION_PHASE:

    eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", "both",
        "--min-step", str(EVAL_MIN_STEP),
        "--n-gen-per-class", str(N_GEN_PER_CLASS),
        "--sample-steps", str(EVAL_SAMPLE_STEPS),
        "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
        "--mini-batch", "1",
        "--inception-batch", str(EVAL_INCEPTION_BATCH),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
    ]
    eval_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
    if EVAL_FORCE_RECOMPUTE:
        eval_cmd.append("--force-recompute")
    if EVAL_DECODE_ON_CPU:
        eval_cmd.append("--decode-on-cpu")
    if EVAL_ECO_TRACK:
        eval_cmd.append("--eco-track")

    add_generation_parallel_args(eval_cmd)
    run_and_stream(eval_cmd, LOGS_DIR / "ldm_evaluate_sdvae.log")

G07_SELECTION_PATH = EXPERIMENT_DIR / "evaluation" / "best_checkpoint.json"
if not G07_SELECTION_PATH.is_file():
    raise FileNotFoundError(f"Frozen G07 selection is unavailable: {G07_SELECTION_PATH}")
with G07_SELECTION_PATH.open(encoding="utf-8") as handle:
    G07_SELECTION = json.load(handle)
recorded_checkpoint = Path(G07_SELECTION["best_checkpoint"])
G07_SELECTED_CHECKPOINT = CHECKPOINTS_DIR / recorded_checkpoint.name
if not G07_SELECTED_CHECKPOINT.is_file() or G07_SELECTED_CHECKPOINT.stat().st_size == 0:
    raise FileNotFoundError(
        f"Frozen G07 periodic checkpoint is unavailable: {G07_SELECTED_CHECKPOINT}"
    )
print("Frozen G07 selection:", G07_SELECTION["best_checkpoint_id"])
print("Portable checkpoint path:", G07_SELECTED_CHECKPOINT)

Frozen G07 selection: step_130000
Portable checkpoint path: /mnt/MammoDiffusion/MammoDiffusion/experiments/diffusers/07_ldm_sdvae_extra1361/checkpoints_ldm/ldm_step130000.keras


## 10. Positive-class generation, filtering, and validation

With the validation-selected checkpoint, the positive pipeline completes 4,083 raw samples and retains 1,361 through the adaptive filter. A generation repair uses `--mode all`, whereas a filter-only repair enters `--mode filter` directly and therefore cannot be blocked by an unnecessary generation-resume check. An independent content-aware `validate` call runs later on every non-plan execution; compatible caches are reused, and missing or stale class-scoped evidence is recomputed. Readable indexed images are reused, corrupt or absent indices are regenerated, and manifests bind the pool to its checkpoint and sampling configuration.

The canonical positive pool is written to `data/synthetic/07_ldm_sdvae_extra1361/positive/`; class-scoped metrics, filter reports, figures, logs, and sustainability events are written under G07 results. The adaptive filter is a quality-control transformation that may exchange diversity for fidelity, and it cannot revise the checkpoint or sampler.


In [11]:
# IDEMPOTENT_GUARD_V1:generation
GEN_N_RAW = 4083
GEN_N_SELECTED = 1361
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = G07_SELECTED_CHECKPOINT
GEN_DECODE_ON_CPU = False
GEN_ECO_TRACK = True
if RUN_GENERATION_PHASE or RUN_FILTER_PHASE:
    POS_EXECUTION_MODE = "all" if RUN_GENERATION_PHASE else "filter"

    pos_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--mode", POS_EXECUTION_MODE,
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", "1",
        "--batch-size", "1",
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
    ]
    pos_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
    if GEN_DECODE_ON_CPU:
        pos_cmd.append("--decode-on-cpu")
    if GEN_ECO_TRACK:
        pos_cmd.append("--eco-track")

    add_generation_parallel_args(pos_cmd)
    run_and_stream(pos_cmd, LOGS_DIR / "ldm_generate_positive_sdvae.log")

## 11. Negative-class generation and two-class completeness

The same checkpoint, SD VAE, sampling budget, guidance scale, raw-pool target, and retained-pool target are applied to label 0. Explicit negative raw and filtered directories prevent collisions with the positive workflow. Generation repair uses `--mode all`, filter-only repair uses `--mode filter`, and independent cache checks provide the corresponding validation artifacts. The final 1,361-image pool is stored at `data/synthetic/07_ldm_sdvae_extra1361/negative/`.

Completing both classes makes G07 eligible for the two-class generator audit. Downstream classifier augmentation uses the benchmark-selected positive representation, whereas the negative pool supports completeness, fidelity, and failure-mode checks. Class-specific comparisons remain descriptive and are not used to reselect G07.


In [12]:
# IDEMPOTENT_GUARD_V1:generation
if RUN_GENERATION_PHASE or RUN_FILTER_PHASE:
    NEG_EXECUTION_MODE = "all" if RUN_GENERATION_PHASE else "filter"
    NEG_RAW_DIR = EXPERIMENT_DIR / "synthetic_raw_negative"
    NEG_FILTERED_DIR = SYNTHETIC_NEW_NEG_DIR

    neg_base_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", "0",
        "--raw-dir", str(NEG_RAW_DIR),
        "--filtered-dir", str(NEG_FILTERED_DIR),
        "--batch-size", "1",
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
    ]
    neg_base_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
    if GEN_DECODE_ON_CPU:
        neg_base_cmd.append("--decode-on-cpu")
    if GEN_ECO_TRACK:
        neg_base_cmd.append("--eco-track")

    add_generation_parallel_args(neg_base_cmd)
    neg_cmd = [*neg_base_cmd, "--mode", NEG_EXECUTION_MODE]
    run_and_stream(neg_cmd, LOGS_DIR / "ldm_negative_sdvae_all.log")

## 12. G07 completion summary

The final cell invokes content-aware validation cache checks for both classes without entering generation, then counts the PNG files in each filtered pool. This provides a concise operational hand-off and should show 1,361 images for each complete canonical pool.

Presence checks do not replace manifest validation. Before using G07 downstream, verify `latents_manifest.json`, `training_manifest.json`, and the class-specific generation/filter manifests. G07's selected status derives from the unified benchmark, not from this summary table alone.

In [13]:
from pathlib import Path
import pandas as pd

def verify_g07_validation_cache(target_label, raw_dir, filtered_dir, log_prefix):
    command = [
        sys.executable, str(UTILITY_DIR / 'generate_ldm.py'), '--project-root', str(PROJECT_ROOT),
        '--experiment-dir', str(EXPERIMENT_DIR), '--model-path', str(GEN_MODEL_PATH),
        '--n-raw', str(GEN_N_RAW), '--n-selected', str(GEN_N_SELECTED),
        '--target-label', str(target_label), '--raw-dir', str(raw_dir),
        '--filtered-dir', str(filtered_dir), '--sample-steps', str(GEN_SAMPLE_STEPS),
        '--guidance-scale', str(GEN_GUIDANCE_SCALE), '--results-stage-name', RESULTS_STAGE_NAME,
        '--vae-backend', 'sd', '--sd-vae-model', str(SD_VAE_MODEL), '--mode', 'validate',
    ]
    if GEN_DECODE_ON_CPU: command.append('--decode-on-cpu')
    if GEN_ECO_TRACK: command.append('--eco-track')
    add_generation_parallel_args(command)
    run_and_stream(command, LOGS_DIR / f'{log_prefix}_validation.log')

if RUN_VALIDATION_PHASE:
    verify_g07_validation_cache(1, EXPERIMENT_DIR / 'synthetic_raw_positive', SYNTHETIC_NEW_POS_DIR, 'ldm_positive_sdvae')
    verify_g07_validation_cache(0, EXPERIMENT_DIR / 'synthetic_raw_negative', SYNTHETIC_NEW_NEG_DIR, 'ldm_negative_sdvae')

pd.DataFrame([
    {'class': name, 'directory': str(directory), 'n_png': len(list(Path(directory).glob('*.png'))) if Path(directory).is_dir() else 0}
    for name, directory in [('positive', SYNTHETIC_NEW_POS_DIR), ('negative', SYNTHETIC_NEW_NEG_DIR)]
])

,class,directory,n_png
0,positive,/mnt/MammoDiffusion/MammoDiffusion/data/synthe...,1361
1,negative,/mnt/MammoDiffusion/MammoDiffusion/data/synthe...,1361
